# Evaluating Agents on Synthetic MCP Benchmarks

Evaluate your agent's tool-use performance across different LLMs using
synthetic benchmark tasks generated by `generate.ipynb`.

### How it works

```
benchmark_tasks.jsonl ──▶ Your Agent (model swapped) ──▶ Model Traces ──▶ Judge ──▶ Rankings
```

The key idea: **the same agent** used for task generation is used for evaluation —
only the underlying LLM is swapped via LangGraph's `configurable` parameter.
This evaluates the full agent stack (tools, guardrails, orchestration), not just
the bare model.

### Prerequisites

1. MCP servers running: `bash start_servers.sh`
2. LangGraph agents running: `bash start_agents.sh`
3. `benchmark_tasks.jsonl` from `generate.ipynb` (or pre-generated)
4. `.env` with API keys

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv
import nest_asyncio
import numpy as np
import pandas as pd

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(NOTEBOOK_DIR / ".env")

from sdg_hub import Flow, FlowRegistry  # noqa: E402

FlowRegistry.discover_flows()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
assert OPENAI_API_KEY and OPENAI_API_KEY != "sk-...", "Set OPENAI_API_KEY in .env"

JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "openai/gpt-4o")

# ── Agent URLs (same agents used for generation) ─────────────────────
AGENT_URLS = {
    "Weather Data": os.environ.get(
        "LANGGRAPH_URL_WEATHER_DATA", "http://localhost:2024"
    ),
    "Medical Calculator": os.environ.get(
        "LANGGRAPH_URL_MEDICAL_CALCULATOR", "http://localhost:2025"
    ),
    "Wikipedia": os.environ.get("LANGGRAPH_URL_WIKIPEDIA", "http://localhost:2026"),
    "Car Price Evaluator": os.environ.get(
        "LANGGRAPH_URL_CAR_PRICE", "http://localhost:2027"
    ),
    "Reddit": os.environ.get("LANGGRAPH_URL_REDDIT", "http://localhost:2028"),
    "DEX Paprika": os.environ.get("LANGGRAPH_URL_DEX_PAPRIKA", "http://localhost:2029"),
}

# ── Models to evaluate ───────────────────────────────────────────────
# Each entry maps a model name to its configurable parameters.
# For OpenAI models: just the model name (agent uses OPENAI_API_KEY from env)
# For local models:  include api_base and api_key
MODEL_CONFIGS = {
    "gpt-5": {},
    "gpt-4o": {},
    "gpt-4o-mini": {},
    # "Qwen3.5-27B": {"api_base": "http://localhost:30000/v1", "api_key": "dummy"},
}

EVAL_MODELS = list(MODEL_CONFIGS.keys())

print(f"Judge:   {JUDGE_MODEL}")
print(f"Models:  {EVAL_MODELS}")
print(f"Agents:  {list(AGENT_URLS.keys())}")

---
## 1. Load Benchmark Tasks

Load tasks from `benchmark_tasks.jsonl` (generated by `generate.ipynb`).

In [ ]:
BENCHMARK_PATH = NOTEBOOK_DIR / "benchmark_tasks.jsonl"
assert BENCHMARK_PATH.exists(), (
    f"No benchmark file at {BENCHMARK_PATH}. Run generate.ipynb first."
)

benchmark_df = pd.read_json(BENCHMARK_PATH, orient="records", lines=True)
print(
    f"Loaded {len(benchmark_df)} tasks across {benchmark_df['server'].nunique()} servers"
)
print()
for server, group in benchmark_df.groupby("server"):
    print(f"  {server:<25} {len(group)} tasks")

---
## 2. Evaluate Models

For each model, run it through the **same LangGraph agent** used for task generation —
only the LLM is swapped via `configurable.model`. The agent's tools, guardrails,
and orchestration logic remain identical.

The evaluation pipeline per model:
1. `AgentBlock` sends tasks to the LangGraph agent with the target model configured
2. `AgentResponseExtractorBlock` extracts text and tool traces from the response
3. Programmatic metrics compare tool usage against the expert gold standard
4. The `Agent Tool-Use Evaluation` flow scores traces via LLM-as-judge (6 dimensions)

In [ ]:
from datasets import Dataset
from eval_utils import (
    ZERO_JUDGE,
    ZERO_RESULT,
    compute_tool_metrics,
    extract_tool_names,
    format_trace_for_judge,
    normalize_tool_trace,
)

from sdg_hub.core.blocks.agent import AgentBlock, AgentResponseExtractorBlock

# Load the judge flow
eval_flow = Flow.from_yaml(FlowRegistry.get_flow_path("Agent Tool-Use Evaluation"))
eval_flow.set_model_config(model=JUDGE_MODEL, api_key=OPENAI_API_KEY)

print("Evaluation pipeline loaded:")
print("  Agent runner:  AgentBlock (LangGraph, model swapped via configurable)")
print(f"  Judge flow:    Agent Tool-Use Evaluation ({len(eval_flow.blocks)} blocks)")
print(f"  Judge model:   {JUDGE_MODEL}")

In [ ]:
RESULTS_PATH = NOTEBOOK_DIR / "evaluation_results.jsonl"

if RESULTS_PATH.exists():
    cached_df = pd.read_json(RESULTS_PATH, orient="records", lines=True)
    all_results = cached_df.to_dict("records")
    cached_combos = set(zip(cached_df["server"], cached_df["model"]))

    # Invalidate cache if benchmark tasks changed (new tasks won't be evaluated otherwise)
    expected_tasks = benchmark_df.groupby("server").size().to_dict()
    cached_tasks = cached_df.groupby("server").size()
    for combo_server in set(s for s, _ in cached_combos):
        expected = expected_tasks.get(combo_server, 0)
        cached = (
            cached_tasks.get(combo_server, 0)
            // cached_df[cached_df["server"] == combo_server]["model"].nunique()
            if combo_server in cached_tasks.index
            else 0
        )
        if expected != cached:
            print(
                f"  Cache stale for {combo_server}: {cached} cached vs {expected} current tasks — will re-evaluate"
            )
            cached_combos = {(s, m) for s, m in cached_combos if s != combo_server}
            all_results = [r for r in all_results if r["server"] != combo_server]

    print(f"Loaded {len(all_results)} cached results")
else:
    all_results = []
    cached_combos = set()

combos_to_run = [
    (s, m)
    for m in EVAL_MODELS
    for s in benchmark_df["server"].unique()
    if (s, m) not in cached_combos
]

if not combos_to_run:
    print(
        f"All {len(EVAL_MODELS)} models x {benchmark_df['server'].nunique()} servers cached."
    )
else:
    print(f"Running {len(combos_to_run)} missing combos")
    task_failures = 0

    for server_name, server_tasks in benchmark_df.groupby("server"):
        server_models = [m for s, m in combos_to_run if s == server_name]
        if not server_models:
            continue

        agent_url = AGENT_URLS.get(server_name)
        if not agent_url:
            # Record zero scores so rankings stay comparable across all servers
            print(f"SKIP {server_name}: no agent URL — recording zero scores")
            for model in server_models:
                for idx in range(len(server_tasks)):
                    all_results.append(
                        {
                            "server": server_name,
                            "model": model,
                            "task_idx": idx,
                            **ZERO_RESULT,
                        }
                    )
            continue

        safe_name = server_name.replace(" ", "_")
        print(f"\n{'=' * 60}")
        print(f"{server_name} ({len(server_tasks)} tasks)")
        print(f"{'=' * 60}")

        for model in server_models:
            print(f"\n  {model}:", end=" ", flush=True)
            config = MODEL_CONFIGS.get(model, {})

            configurable = {"model": model}
            if "api_base" in config:
                configurable["api_base"] = config["api_base"]
            if "api_key" in config:
                configurable["api_key"] = config["api_key"]

            agent_block = AgentBlock(
                block_name=f"eval_{safe_name}",
                agent_framework="langgraph",
                agent_url=agent_url,
                timeout=300.0,
                input_cols=["question"],
                output_cols=["agent_response"],
                connector_kwargs={"run_config": {"configurable": configurable}},
            )

            extractor = AgentResponseExtractorBlock(
                block_name=f"extract_{safe_name}",
                agent_framework="langgraph",
                input_cols="agent_response",
                extract_text=True,
                extract_tool_trace=True,
            )

            # Step 1: Run model through agent
            try:
                result_df = agent_block.generate(server_tasks[["question"]].copy())
                result_df = extractor.generate(result_df)
            except Exception as e:
                print(f"FAILED ({e})")
                task_failures += len(server_tasks)
                for idx in range(len(server_tasks)):
                    all_results.append(
                        {
                            "server": server_name,
                            "model": model,
                            "task_idx": idx,
                            **ZERO_RESULT,
                        }
                    )
                continue

            # Step 2: Extract traces, compute metrics, prepare judge input
            text_col = f"extract_{safe_name}_text"
            trace_col = f"extract_{safe_name}_tool_trace"
            judge_rows = []
            task_meta = []

            for idx in range(len(server_tasks)):
                task = server_tasks.iloc[idx]

                m_answer = str(result_df[text_col].iloc[idx] or "")
                m_trace_raw = result_df[trace_col].iloc[idx]
                m_trace_raw = m_trace_raw if isinstance(m_trace_raw, list) else []

                m_trace = normalize_tool_trace(m_trace_raw)
                m_tools = extract_tool_names(m_trace)

                e_tools = task["expert_tools"]
                e_trace = task["expert_tool_trace"]
                metrics = compute_tool_metrics(m_tools, e_tools, m_trace, e_trace)

                judge_rows.append(
                    {
                        "question": task["question"],
                        "expert_answer_truncated": task["expert_answer"][:2000],
                        "expert_trace_formatted": format_trace_for_judge(e_trace),
                        "model_answer": m_answer[:2000],
                        "model_trace_formatted": format_trace_for_judge(m_trace),
                    }
                )
                task_meta.append(
                    {
                        "server": server_name,
                        "model": model,
                        "task_idx": idx,
                        **metrics,
                    }
                )

            # Step 3: Run judge flow
            if judge_rows:
                try:
                    judge_ds = Dataset.from_list(judge_rows)
                    judge_result = eval_flow.generate(judge_ds)
                    judge_df = (
                        judge_result.to_pandas()
                        if hasattr(judge_result, "to_pandas")
                        else pd.DataFrame(judge_result)
                    )
                    judge_cols = [
                        "task_fulfillment",
                        "grounding",
                        "tool_appropriateness",
                        "parameter_accuracy",
                        "dependency_awareness",
                        "parallelism_and_efficiency",
                    ]
                    for i, meta in enumerate(task_meta):
                        scores = {}
                        for col in judge_cols:
                            val = (
                                judge_df[col].iloc[i] if col in judge_df.columns else 0
                            )
                            try:
                                scores[col] = int(val)
                            except (ValueError, TypeError):
                                scores[col] = 0
                        all_results.append({**meta, **scores})
                except Exception as e:
                    print(f"\n    Judge failed: {e}")
                    task_failures += len(task_meta)
                    for meta in task_meta:
                        all_results.append({**meta, **ZERO_JUDGE})

            n = sum(
                1
                for r in all_results
                if r["model"] == model and r["server"] == server_name
            )
            avg = np.mean(
                [
                    r.get("task_fulfillment", 0)
                    for r in all_results
                    if r["model"] == model and r["server"] == server_name
                ]
            )
            print(f"{n} tasks, avg_fulfillment={avg:.1f}")

    if task_failures:
        print(f"\nWARNING: {task_failures} task(s) failed — scored 0.")

    pd.DataFrame(all_results).to_json(RESULTS_PATH, orient="records", lines=True)
    print(f"\nSaved {len(all_results)} results to {RESULTS_PATH.name}")

print(f"\nTotal: {len(all_results)} evaluation scores")

---
## 3. Results

In [ ]:
results_df = pd.DataFrame(all_results)

judge_cols = [
    "task_fulfillment",
    "grounding",
    "tool_appropriateness",
    "parameter_accuracy",
    "dependency_awareness",
    "parallelism_and_efficiency",
]
tool_cols = ["tool_recall", "tool_precision", "order_match", "param_match"]
all_cols = tool_cols + judge_cols

server_model = results_df.groupby(["server", "model"])[all_cols].mean()
for col in judge_cols:
    server_model[col] = server_model[col] / 10.0
server_model["overall"] = server_model.mean(axis=1)

pivot = server_model["overall"].unstack("model")
col_order = pivot.mean().sort_values(ascending=False).index.tolist()
pivot = pivot[col_order]

n_models = results_df["model"].nunique()
task_counts = results_df.groupby("server").size() // n_models
pivot.insert(0, "tasks", task_counts)

pivot.loc["OVERALL"] = pivot.mean()
pivot.loc["OVERALL", "tasks"] = pivot.loc[pivot.index != "OVERALL", "tasks"].sum()

server_rows = pivot.drop("OVERALL").sort_values(col_order[0], ascending=False)
pivot = pd.concat([server_rows, pivot.loc[["OVERALL"]]])

print("Overall Score by Server x Model (per-server averages)")
print("=" * 80)
print(pivot.round(3).to_string())

ranking = sorted(col_order, key=lambda m: pivot.loc["OVERALL", m], reverse=True)
print(f"\nRanking: {' > '.join(ranking)}")

# Dimension breakdown
print("\nDimension breakdown:")
dim_summary = results_df.groupby("model")[judge_cols].mean() / 10.0
dim_summary = dim_summary.loc[[m for m in ranking if m in dim_summary.index]]
dim_summary["task_completion"] = dim_summary[["task_fulfillment", "grounding"]].mean(
    axis=1
)
dim_summary["tool_selection"] = dim_summary[
    ["tool_appropriateness", "parameter_accuracy"]
].mean(axis=1)
dim_summary["planning"] = dim_summary[
    ["dependency_awareness", "parallelism_and_efficiency"]
].mean(axis=1)
print(
    dim_summary[["task_completion", "tool_selection", "planning"]].round(3).to_string()
)

results_df.to_json(RESULTS_PATH, orient="records", lines=True)
print(f"\nSaved {len(results_df)} results to {RESULTS_PATH.name}")